**Chapter 19 – Training and Deploying TensorFlow Models at Scale**

_This notebook contains all the sample code and solutions to the exercises in chapter 19._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/19_training_and_deploying_at_scale.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.7 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 7)

**Warning**: the latest TensorFlow versions are based on Keras 3. For chapters 10-15, it wasn't too hard to update the code to support Keras 3, but unfortunately it's much harder for this chapter, so I've had to revert to Keras 2. To do that, I set the `TF_USE_LEGACY_KERAS` environment variable to `"1"` and import the `tf_keras` package. This ensures that `tf.keras` points to `tf_keras`, which is Keras 2.*.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras

And TensorFlow ≥ 2.8:

In [ ]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

If running on Colab or Kaggle, you need to install the Google AI Platform client library, which will be used later in this notebook. You can ignore the warnings about version incompatibilities.

* **Warning**: On Colab, you must restart the Runtime after the installation, and continue with the next cells.

In [ ]:
import sys
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:
    %pip install -q -U google-cloud-aiplatform

This chapter discusses how to run or train a model on one or more GPUs, so let's make sure there's at least one, or else issue a warning:

In [ ]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

# Serving a TensorFlow Model

Let's start by deploying a model using TF Serving, then we'll deploy to Google Vertex AI.

## Using TensorFlow Serving

The first thing we need to do is to build and train a model, and export it to the SavedModel format.

### Exporting SavedModels

Let's load the MNIST dataset, scale it, and split it.

In [ ]:
from pathlib import Path
import tensorflow as tf

# extra code – load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – build & train an MNIST model (also handles image preprocessing)
tf.random.set_seed(42)
tf.keras.backend.clear_session()
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),
    tf.keras.layers.Rescaling(scale=1 / 255),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

model_name = "my_mnist_model"
model_version = "0001"
model_path = Path(model_name) / model_version
model.save(model_path, save_format="tf")

Epoch 1/10
1719/1719 [==============================] - 2s 1ms/step - loss: 0.7012 - accuracy: 0.8241 - val_loss: 0.3715 - val_accuracy: 0.9024
Epoch 2/10
1719/1719 [==============================] - 2s 943us/step - loss: 0.3536 - accuracy: 0.9020 - val_loss: 0.2990 - val_accuracy: 0.9144
Epoch 3/10
1719/1719 [==============================] - 2s 933us/step - loss: 0.3036 - accuracy: 0.9145 - val_loss: 0.2651 - val_accuracy: 0.9272
Epoch 4/10
1719/1719 [==============================] - 2s 965us/step - loss: 0.2736 - accuracy: 0.9231 - val_loss: 0.2436 - val_accuracy: 0.9334
Epoch 5/10
1719/1719 [==============================] - 2s 946us/step - loss: 0.2509 - accuracy: 0.9296 - val_loss: 0.2257 - val_accuracy: 0.9364
Epoch 6/10
1719/1719 [==============================] - 2s 974us/step - loss: 0.2322 - accuracy: 0.9350 - val_loss: 0.2121 - val_accuracy: 0.9396
Epoch 7/10
1719/1719 [==============================] - 2s 959us/step - loss: 0.2161 - accuracy: 0.9400 - val_loss: 0.1970 - v

Let's take a look at the file tree (we've discussed what each of these file is used for in chapter 10):

In [ ]:
sorted([str(path) for path in model_path.parent.glob("**/*")])  # extra code

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/keras_metadata.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index']

Let's inspect the SavedModel:

In [ ]:
!saved_model_cli show --dir '{model_path}'

The given SavedModel contains the following tag-sets:
'serve'


In [ ]:
!saved_model_cli show --dir '{model_path}' --tag_set serve

The given SavedModel MetaGraphDef contains SignatureDefs with the following keys:
SignatureDef key: "__saved_model_init_op"
SignatureDef key: "serving_default"


In [ ]:
!saved_model_cli show --dir '{model_path}' --tag_set serve \
                      --signature_def serving_default

The given SavedModel SignatureDef contains the following input(s):
  inputs['flatten_input'] tensor_info:
      dtype: DT_UINT8
      shape: (-1, 28, 28)
      name: serving_default_flatten_input:0
The given SavedModel SignatureDef contains the following output(s):
  outputs['dense_1'] tensor_info:
      dtype: DT_FLOAT
      shape: (-1, 10)
      name: StatefulPartitionedCall:0
Method name is: tensorflow/serving/predict


For even more details, you can run the following command:

```ipython
!saved_model_cli show --dir '{model_path}' --all
```

### Installing and Starting TensorFlow Serving

If you are running this notebook in Colab or Kaggle, TensorFlow Server needs to be installed:

In [ ]:
if "google.colab" in sys.modules or "kaggle_secrets" in sys.modules:
    url = "https://storage.googleapis.com/tensorflow-serving-apt"
    src = "stable tensorflow-model-server tensorflow-model-server-universal"
    !echo 'deb {url} {src}' > /etc/apt/sources.list.d/tensorflow-serving.list
    !curl '{url}/tensorflow-serving.release.pub.gpg' | apt-key add -
    !apt update -q && apt-get install -y tensorflow-model-server
    %pip install -q -U tensorflow-serving-api

If `tensorflow_model_server` is installed (e.g., if you are running this notebook in Colab), then the following 2 cells will start the server. If your OS is Windows, you may need to run the `tensorflow_model_server` command in a terminal, and replace `${MODEL_DIR}` with the full path to the `my_mnist_model` directory.

In [ ]:
import os

os.environ["MODEL_DIR"] = str(model_path.parent.absolute())

In [ ]:
%%bash --bg
tensorflow_model_server \
    --port=8500 \
    --rest_api_port=8501 \
    --model_name=my_mnist_model \
    --model_base_path="${MODEL_DIR}" >my_server.log 2>&1

In [ ]:
import time

time.sleep(2) # let's wait a couple seconds for the server to start

If you are running this notebook on your own machine, and you prefer to install TF Serving using Docker, first make sure [Docker](https://docs.docker.com/install/) is installed, then run the following commands in a terminal. You must replace `/path/to/my_mnist_model` with the appropriate absolute path to the `my_mnist_model` directory, but do not modify the container path `/models/my_mnist_model`.

```bash
docker pull tensorflow/serving  # downloads the latest TF Serving image

docker run -it --rm -v "/path/to/my_mnist_model:/models/my_mnist_model" \
    -p 8500:8500 -p 8501:8501 -e MODEL_NAME=my_mnist_model tensorflow/serving
```

### Querying TF Serving through the REST API

Next, let's send a REST query to TF Serving:

In [ ]:
import json

X_new = X_test[:3]  # pretend we have 3 new digit images to classify
request_json = json.dumps({
    "signature_name": "serving_default",
    "instances": X_new.tolist(),
})

In [ ]:
request_json[:100] + "..." + request_json[-10:]

'{"signature_name": "serving_default", "instances": [[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..., 0, 0]]]}'

Now let's use TensorFlow Serving's REST API to make predictions:

In [ ]:
import requests

server_url = "http://localhost:8501/v1/models/my_mnist_model:predict"
response = requests.post(server_url, data=request_json)
response.raise_for_status()  # raise an exception in case of error
response = response.json()

In [ ]:
import numpy as np

y_proba = np.array(response["predictions"])
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

### Querying TF Serving through the gRPC API

In [ ]:
from tensorflow_serving.apis.predict_pb2 import PredictRequest

request = PredictRequest()
request.model_spec.name = model_name
request.model_spec.signature_name = "serving_default"
input_name = model.input_names[0]  # == "flatten_input"
request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new))

In [ ]:
import grpc
from tensorflow_serving.apis import prediction_service_pb2_grpc

channel = grpc.insecure_channel('localhost:8500')
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel)
response = predict_service.Predict(request, timeout=10.0)

Convert the response to a tensor:

In [ ]:
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
y_proba = tf.make_ndarray(outputs_proto)

In [ ]:
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]],
      dtype=float32)

If your client does not include the TensorFlow library, you can convert the response to a NumPy array like this:

In [ ]:
# extra code – shows how to avoid using tf.make_ndarray()
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
shape = [dim.size for dim in outputs_proto.tensor_shape.dim]
y_proba = np.array(outputs_proto.float_val).reshape(shape)
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

### Deploying a new model version

In [ ]:
# extra code – build and train a new MNIST model version
np.random.seed(42)
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8),
    tf.keras.layers.Rescaling(scale=1 / 255),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

Epoch 1/10
1719/1719 [==============================] - 2s 931us/step - loss: 0.7039 - accuracy: 0.8056 - val_loss: 0.3418 - val_accuracy: 0.9042
Epoch 2/10
1719/1719 [==============================] - 1s 855us/step - loss: 0.3204 - accuracy: 0.9082 - val_loss: 0.2674 - val_accuracy: 0.9242
Epoch 3/10
1719/1719 [==============================] - 2s 883us/step - loss: 0.2650 - accuracy: 0.9235 - val_loss: 0.2227 - val_accuracy: 0.9368
Epoch 4/10
1719/1719 [==============================] - 1s 869us/step - loss: 0.2319 - accuracy: 0.9329 - val_loss: 0.2032 - val_accuracy: 0.9432
Epoch 5/10
1719/1719 [==============================] - 1s 870us/step - loss: 0.2089 - accuracy: 0.9399 - val_loss: 0.1833 - val_accuracy: 0.9482
Epoch 6/10
1719/1719 [==============================] - 1s 871us/step - loss: 0.1908 - accuracy: 0.9446 - val_loss: 0.1740 - val_accuracy: 0.9498
Epoch 7/10
1719/1719 [==============================] - 2s 873us/step - loss: 0.1756 - accuracy: 0.9490 - val_loss: 0.1605 -

In [ ]:
model_version = "0002"
model_path = Path(model_name) / model_version
model.save(model_path, save_format="tf")

INFO:tensorflow:Assets written to: my_mnist_model/0002/assets


Let's take a look at the file tree again:

In [ ]:
sorted([str(path) for path in model_path.parent.glob("**/*")])  # extra code

['my_mnist_model/0001',
 'my_mnist_model/0001/assets',
 'my_mnist_model/0001/keras_metadata.pb',
 'my_mnist_model/0001/saved_model.pb',
 'my_mnist_model/0001/variables',
 'my_mnist_model/0001/variables/variables.data-00000-of-00001',
 'my_mnist_model/0001/variables/variables.index',
 'my_mnist_model/0002',
 'my_mnist_model/0002/assets',
 'my_mnist_model/0002/keras_metadata.pb',
 'my_mnist_model/0002/saved_model.pb',
 'my_mnist_model/0002/variables',
 'my_mnist_model/0002/variables/variables.data-00000-of-00001',
 'my_mnist_model/0002/variables/variables.index']

**Warning**: You may need to wait a minute before the new model is loaded by TensorFlow Serving.

In [ ]:
import requests

server_url = "http://localhost:8501/v1/models/my_mnist_model:predict"

response = requests.post(server_url, data=request_json)
response.raise_for_status()
response = response.json()

In [ ]:
response.keys()

dict_keys(['predictions'])

In [ ]:
y_proba = np.array(response["predictions"])
y_proba.round(2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.99, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ]])

## Creating a Prediction Service on Vertex AI

Follow the instructions in the book to create a Google Cloud Platform account and activate the Vertex AI and Cloud Storage APIs. Then, if you're running this notebook in Colab, you can run the following cell to authenticate using the same Google account as you used with Google Cloud Platform, and authorize this Colab to access your data.

**WARNING: only do this if you trust this notebook!**
* Be extra careful if this is not the official notebook from https://github.com/ageron/handson-ml3: the Colab URL should start with https://colab.research.google.com/github/ageron/handson-ml3. Or else, the code could do whatever it wants with your data.

If you are not running this notebook in Colab, you must follow the instructions in the book to create a service account and generate a key for it, download it to this notebook's directory, and name it `my_service_account_key.json` (or make sure the `GOOGLE_APPLICATION_CREDENTIALS` environment variable points to your key).

In [ ]:
project_id = "my_project"  ##### CHANGE THIS TO YOUR PROJECT ID #####

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif "kaggle_secrets" in sys.modules:
    from kaggle_secrets import UserSecretsClient
    UserSecretsClient().set_gcloud_credentials(project=project_id)
else:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my_service_account_key.json"

In [ ]:
from google.cloud import storage

bucket_name = "my_bucket"  ##### CHANGE THIS TO A UNIQUE BUCKET NAME #####
location = "us-central1"

storage_client = storage.Client(project=project_id)
bucket = storage_client.create_bucket(bucket_name, location=location)
#bucket = storage_client.bucket(bucket_name)  # to reuse a bucket instead

In [ ]:
def upload_directory(bucket, dirpath):
    dirpath = Path(dirpath)
    for filepath in dirpath.glob("**/*"):
        if filepath.is_file():
            blob = bucket.blob(filepath.relative_to(dirpath.parent).as_posix())
            blob.upload_from_filename(filepath)

upload_directory(bucket, "my_mnist_model")

In [ ]:
# extra code – a much faster multithreaded implementation of upload_directory()
#              which also accepts a prefix for the target path, and prints stuff

from concurrent import futures

def upload_file(bucket, filepath, blob_path):
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(filepath)

def upload_directory(bucket, dirpath, prefix=None, max_workers=50):
    dirpath = Path(dirpath)
    prefix = prefix or dirpath.name
    with futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_filepath = {
            executor.submit(
                upload_file,
                bucket, filepath,
                f"{prefix}/{filepath.relative_to(dirpath).as_posix()}"
            ): filepath
            for filepath in sorted(dirpath.glob("**/*"))
            if filepath.is_file()
        }
        for future in futures.as_completed(future_to_filepath):
            filepath = future_to_filepath[future]
            try:
                result = future.result()
            except Exception as ex:
                print(f"Error uploading {filepath!s:60}: {ex}")  # f!s is str(f)
            else:
                print(f"Uploaded {filepath!s:60}", end="\r")

    print(f"Uploaded {dirpath!s:60}")

Alternatively, if you installed Google Cloud CLI (it's preinstalled on Colab), then you can use the following `gsutil` command:

In [ ]:
#!gsutil -m cp -r my_mnist_model gs://{bucket_name}/

In [ ]:
from google.cloud import aiplatform

server_image = "gcr.io/cloud-aiplatform/prediction/tf2-gpu.2-8:latest"

aiplatform.init(project=project_id, location=location)
mnist_model = aiplatform.Model.upload(
    display_name="mnist",
    artifact_uri=f"gs://{bucket_name}/my_mnist_model/0001",
    serving_container_image_uri=server_image,
)

Creating Model
Create Model backing LRO: projects/522977795627/locations/us-central1/models/4798114811986575360/operations/53403898236370944
Model created. Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360
To use this Model in another session:
model = aiplatform.Model('projects/522977795627/locations/us-central1/models/4798114811986575360')


**Warning**: this cell may take several minutes to run, as it waits for Vertex AI to provision the compute nodes:

In [ ]:
endpoint = aiplatform.Endpoint.create(display_name="mnist-endpoint")

endpoint.deploy(
    mnist_model,
    min_replica_count=1,
    max_replica_count=5,
    machine_type="n1-standard-4",
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=1
)

Creating Endpoint
Create Endpoint backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/4135354010494304256
Endpoint created. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/522977795627/locations/us-central1/endpoints/5133373499481522176')
Deploying Model projects/522977795627/locations/us-central1/models/4798114811986575360 to Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/388359120522051584
Endpoint model deployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


In [ ]:
response = endpoint.predict(instances=X_new.tolist())

In [ ]:
import numpy as np

np.round(response.predictions, 2)

array([[0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 0.99, 0.01, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.97, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01, 0.  , 0.  ]])

In [ ]:
endpoint.undeploy_all()  # undeploy all models from the endpoint
endpoint.delete()

Undeploying Endpoint model: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Undeploy Endpoint model backing LRO: projects/522977795627/locations/us-central1/endpoints/5133373499481522176/operations/3579722406467469312
Endpoint model undeployed. Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Deleting Endpoint : projects/522977795627/locations/us-central1/endpoints/5133373499481522176
Delete Endpoint  backing LRO: projects/522977795627/locations/us-central1/operations/4738836360561950720
Endpoint deleted. . Resource name: projects/522977795627/locations/us-central1/endpoints/5133373499481522176


## Running Batch Prediction Jobs on Vertex AI

In [ ]:
batch_path = Path("my_mnist_batch")
batch_path.mkdir(exist_ok=True)
with open(batch_path / "my_mnist_batch.jsonl", "w") as jsonl_file:
    for image in X_test[:100].tolist():
        jsonl_file.write(json.dumps(image))
        jsonl_file.write("\n")

upload_directory(bucket, batch_path)

Uploaded my_mnist_batch                                              


In [ ]:
batch_prediction_job = mnist_model.batch_predict(
    job_display_name="my_batch_prediction_job",
    machine_type="n1-standard-4",
    starting_replica_count=1,
    max_replica_count=5,
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=1,
    gcs_source=[f"gs://{bucket_name}/{batch_path.name}/my_mnist_batch.jsonl"],
    gcs_destination_prefix=f"gs://{bucket_name}/my_mnist_predictions/",
    sync=True  # set to False if you don't want to wait for completion
)

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/4346926367237996544?project=522977795627
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_PENDING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/522977795627/locations/us-central1/batchPredictionJobs/

In [ ]:
batch_prediction_job.output_info  # extra code – shows the output directory

gcs_output_directory: "gs://my_bucket/my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z"

In [ ]:
y_probas = []
for blob in batch_prediction_job.iter_outputs():
    print(blob.name)  # extra code
    if "prediction.results" in blob.name:
        for line in blob.download_as_text().splitlines():
            y_proba = json.loads(line)["prediction"]
            y_probas.append(y_proba)

my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.errors_stats-00000-of-00001
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00000-of-00002
my_mnist_predictions/prediction-mnist-2022_04_12T21_30_08_071Z/prediction.results-00001-of-00002


In [ ]:
y_pred = np.argmax(y_probas, axis=1)
accuracy = np.sum(y_pred == y_test[:100]) / 100

In [ ]:
accuracy

0.98

In [ ]:
mnist_model.delete()

Deleting Model : projects/522977795627/locations/us-central1/models/4798114811986575360
Delete Model  backing LRO: projects/522977795627/locations/us-central1/operations/598902403101622272
Model deleted. . Resource name: projects/522977795627/locations/us-central1/models/4798114811986575360


Let's delete all the directories we created on GCS (i.e., all the blobs with these prefixes):

In [ ]:
for prefix in ["my_mnist_model/", "my_mnist_batch/", "my_mnist_predictions/"]:
    blobs = bucket.list_blobs(prefix=prefix)
    for blob in blobs:
        blob.delete()

#bucket.delete()  # uncomment and run if you want to delete the bucket itself
batch_prediction_job.delete()

Deleting BatchPredictionJob : projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544
Delete BatchPredictionJob  backing LRO: projects/522977795627/locations/us-central1/operations/6699028098374959104
BatchPredictionJob deleted. . Resource name: projects/522977795627/locations/us-central1/batchPredictionJobs/4346926367237996544


# Deploying a Model to a Mobile or Embedded Device

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(str(model_path))
tflite_model = converter.convert()
with open("my_converted_savedmodel.tflite", "wb") as f:
    f.write(tflite_model)

2022-04-10 09:03:52.237094: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:03:52.237108: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:03:52.237830: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: my_mnist_model/0001
2022-04-10 09:03:52.238869: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:03:52.238881: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: my_mnist_model/0001
2022-04-10 09:03:52.242108: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:03:52.263868: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: my_mnist_model/0001
2022-04-10 09:03:52.271298: I tensorflow/cc/saved_model/loader.cc:301] SavedModel load for tags { serve }; Status: success: OK. Too

In [ ]:
# extra code – shows how to convert a Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]

In [ ]:
tflite_model = converter.convert()
with open("my_converted_keras_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets


INFO:tensorflow:Assets written to: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs/assets
2022-04-10 09:26:30.319286: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2022-04-10 09:26:30.319301: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2022-04-10 09:26:30.319417: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.320420: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2022-04-10 09:26:30.320431: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: /var/folders/wy/h39t6kb11pnbb0pzhksd_fqh0000gq/T/tmp6ffbc1qs
2022-04-10 09:26:30.323773: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2022-04-10 09:26:30.345416: I tensorflow/cc/saved_model/loader.cc:212] Running initialization

# Running a Model in a Web Page

Code examples for this section are hosted on glitch.com, a website that lets you create Web apps for free.

* https://homl.info/tfjscode: a simple TFJS Web app that loads a pretrained model and classifies an image.
* https://homl.info/tfjswpa: the same Web app setup as a WPA. Try opening this link on various platforms, including mobile devices.
** https://homl.info/wpacode: this WPA's source code.
* https://tensorflow.org/js: The TFJS library.
** https://www.tensorflow.org/js/demos: some fun demos.

<details>
<summary><b> ✅AI Understanding Template-ch0_01_Please provide answer for AI w.r.t. code explanation:
    • What is it?
    • How does it reason?
    • Where does it fail?
    • When should I use it?
    • What is the mental model?
    • How do I prompt it?
    • What are alternatives? </b>
</summary>
Below is a **1-page AI Understanding Template (Interview-ready)** for **Chapter 19: Training & Deploying TensorFlow Models at Scale**.
(Concise, conceptual, tied to the code you shared.)

---

## 🧠 AI Understanding Template – Training & Deploying TensorFlow Models at Scale

### 1️⃣ What is it?

A **production ML pipeline** that:

* Trains TensorFlow models
* Exports them as **SavedModel**
* Deploys them using **TensorFlow Serving**, **Vertex AI**, **batch prediction**, **edge (TFLite)**, and **web (TFJS)**

👉 Focus: **model lifecycle beyond training**

---

### 2️⃣ How does it reason?

The system reasons via:

* **Trained neural networks** (MNIST classifier)
* **Forward pass inference** (no learning at serving time)
* **Stateless prediction APIs** (REST / gRPC)
* **Versioned models** (`0001`, `0002`) with hot-swapping

Reasoning is **numerical + deterministic**, not symbolic.

---

### 3️⃣ Where does it fail?

* ❌ Poor monitoring → silent accuracy decay
* ❌ Data drift not handled automatically
* ❌ TF Serving assumes strict input schema
* ❌ High latency without GPU / autoscaling
* ❌ Complex infra setup (IAM, buckets, endpoints)

---

### 4️⃣ When should I use it?

Use this approach when:

* You need **real-time inference at scale**
* Multiple clients (web, mobile, batch) consume the model
* You want **versioning, rollback, autoscaling**
* You deploy to **cloud or edge devices**

Not ideal for quick experiments.

---

### 5️⃣ What is the mental model?

Think in **layers**:

```
Training → SavedModel → Serving Layer → Client
```

or

```
Model Artifact → API Endpoint → Prediction
```

Key idea:
👉 *Training and inference are decoupled*

---

### 6️⃣ How do I prompt it? (Operational Thinking)

You don’t “prompt” it like an LLM.
You **interact via APIs**:

* REST JSON:

```json
{
  "instances": [[28x28 image]]
}
```

* gRPC tensors
* Batch JSONL files
* TFLite byte input (mobile)
* TFJS tensors (browser)

Correct **input shape + dtype = success**

---

### 7️⃣ What are alternatives?

| Purpose             | Alternative    |
| ------------------- | -------------- |
| Serving             | FastAPI + ONNX |
| Cloud ML            | AWS SageMaker  |
| Lightweight serving | BentoML        |
| Edge deployment     | CoreML         |
| Web ML              | ONNX.js        |
| Experiment tracking | MLflow         |

---

## 🎯 Interview One-Liner

> “This code shows how to move a TensorFlow model from training to scalable production using SavedModel, TensorFlow Serving, Vertex AI, batch jobs, edge deployment, and web inference.”

If you want next:

* ✅ **Flow diagram**
* ✅ **Key terms glossary**
* ✅ **MCQs or scenario questions**
* ✅ **Production interview answers**
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_02_Explain the code, get key terms and provide their full definition as well</b>
</summary>
Below is a **structured, interview-ready AI understanding template (Ch-19)**.
It explains **what the code does**, **why it’s used**, and **key terms with full definitions**.

---

## 1️⃣ High-Level Purpose (What this chapter does)

This code demonstrates **training, versioning, serving, and deploying TensorFlow models at scale** using:

* **TensorFlow Serving**
* **REST & gRPC APIs**
* **Google Vertex AI**
* **Batch prediction**
* **Edge deployment (TFLite)**
* **Web deployment (TF.js)**

---

## 2️⃣ Setup & Environment Control

### What the code does

* Enforces **Python ≥ 3.7** and **TensorFlow ≥ 2.8**
* Forces **Keras 2** compatibility using:

```python
os.environ["TF_USE_LEGACY_KERAS"] = "1"
```

* Checks GPU availability
* Installs Google Cloud & Vertex AI libraries (Colab/Kaggle)

### Key Terms

* **Legacy Keras**: Older Keras API (2.x) required for backward compatibility.
* **GPU acceleration**: Using GPUs to speed up neural network training/inference.

---

## 3️⃣ Model Training (MNIST)

### What happens

* Loads MNIST dataset
* Preprocesses images (Flatten + Rescaling)
* Trains a **Dense neural network**
* Saves model in **SavedModel format**

```python
model.save(model_path, save_format="tf")
```

### Key Terms

* **SavedModel**: TensorFlow’s standard, production-ready model format.
* **Sparse categorical cross-entropy**: Loss function for integer-encoded class labels.
* **Softmax**: Converts logits into probability distribution.

---

## 4️⃣ TensorFlow Serving (Production Inference)

### What happens

* Starts **TensorFlow Model Server**
* Loads model by **name + version**
* Exposes:

  * **REST API (8501)**
  * **gRPC API (8500)**

### Key Terms

* **TensorFlow Serving**: High-performance model serving system for production.
* **Model versioning**: Multiple model versions served simultaneously.
* **SignatureDef**: Defines model inputs & outputs for serving.

---

## 5️⃣ REST API Prediction

### What happens

* Sends JSON payload to TF Serving
* Receives predictions via HTTP

```json
{
  "instances": [...]
}
```

### Key Terms

* **REST API**: HTTP-based communication using JSON.
* **Inference**: Running a trained model to make predictions.

---

## 6️⃣ gRPC Prediction

### What happens

* Uses binary protocol (faster than REST)
* Uses `PredictRequest` and protobufs

### Key Terms

* **gRPC**: High-performance RPC framework using Protocol Buffers.
* **Protocol Buffers**: Binary serialization format by Google.

---

## 7️⃣ Model Version Upgrade

### What happens

* Trains **version 0002**
* TF Serving automatically switches to latest version

### Key Terms

* **Canary deployment**: Gradually rolling out new model versions.
* **Backward compatibility**: New versions don’t break clients.

---

## 8️⃣ Vertex AI Online Prediction

### What happens

* Uploads model to **Google Cloud Storage**
* Deploys model to **Vertex AI endpoint**
* Scales with replicas & GPUs

### Key Terms

* **Vertex AI**: Google’s managed ML platform.
* **Endpoint**: HTTPS endpoint for online predictions.
* **Artifact URI**: GCS path where model is stored.

---

## 9️⃣ Batch Prediction (Offline)

### What happens

* Writes test data as `.jsonl`
* Runs batch job on Vertex AI
* Collects predictions from GCS

### Key Terms

* **Batch prediction**: Offline, large-scale inference.
* **JSONL**: JSON Lines format (one JSON per line).
* **Autoscaling**: Dynamically adjusts compute resources.

---

## 🔟 Edge & Mobile Deployment (TFLite)

### What happens

* Converts model to `.tflite`
* Applies optimization

```python
converter.optimizations = [tf.lite.Optimize.DEFAULT]
```

### Key Terms

* **TensorFlow Lite**: Lightweight runtime for mobile/embedded devices.
* **Quantization**: Reduces model size & latency.

---

## 1️⃣1️⃣ Web Deployment (TF.js)

### What it enables

* Run models **directly in browser**
* No server required

### Key Terms

* **TensorFlow.js**: JavaScript ML library for browsers.
* **Client-side inference**: Prediction runs on user’s device.

---

## 🎯 Interview Summary (One-liner)

> “This code demonstrates an end-to-end ML lifecycle: training a TensorFlow model, serving it with TensorFlow Serving, deploying it to Vertex AI for online and batch inference, and optimizing it for edge and web deployment.”

If you want next:

* ✅ **1-page cheat sheet**
* ✅ **MCQs**
* ✅ **Scenario-based interview Q&A**
* ✅ **Architecture diagram explanation**
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_03_cheatsheet</b>
</summary>
Below is a **one-page AI Understanding Cheat Sheet (Chapter 19)** focused on **code explanation + key concepts**, written **interview-ready and concise**.

---

# 🧠 AI Understanding Template – Chapter 19

## **Training & Deploying TensorFlow Models at Scale**

---

## 🎯 Goal of This Chapter

Learn **how to train, export, serve, and deploy TensorFlow models at scale** using:

* **TensorFlow Serving**
* **REST / gRPC APIs**
* **Google Vertex AI**
* **Batch Prediction**
* **Mobile (TFLite) & Web (TFJS)**

---

## 1️⃣ Environment & Compatibility Setup

```python
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras
```

### Why?

* Forces **Keras 2.x** for backward compatibility.
* Required because newer Keras 3 breaks older training & serving APIs.

**Key Terms**

* **Legacy Keras**: Older stable Keras version compatible with TF Serving
* **Reproducibility**: `tf.random.set_seed(42)`

---

## 2️⃣ Model Training (MNIST)

```python
model = tf.keras.Sequential([...])
model.compile(...)
model.fit(...)
```

### What Happens?

* Train a **digit classifier**
* Includes **preprocessing inside the model** (Flatten + Rescaling)

**Key Terms**

* **End-to-end model**: Handles preprocessing + inference
* **Softmax**: Converts logits → class probabilities
* **SGD**: Gradient-based optimizer

---

## 3️⃣ SavedModel Export

```python
model.save("my_mnist_model/0001", save_format="tf")
```

### Why SavedModel?

* Required for **TensorFlow Serving**
* Versioned folders enable **model versioning**

**Key Terms**

* **SavedModel**: TensorFlow’s production format
* **Model Versioning**: Multiple versions served simultaneously
* **Signature**: Defines model inputs & outputs

---

## 4️⃣ TensorFlow Serving

```bash
tensorflow_model_server \
 --model_name=my_mnist_model \
 --model_base_path=MODEL_DIR
```

### What It Does?

* Runs a **high-performance inference server**
* Supports **REST & gRPC**

**Key Terms**

* **Model Server**: Dedicated inference engine
* **Hot Reloading**: New versions auto-loaded
* **Scalability**: Handles high-traffic inference

---

## 5️⃣ REST API Inference

```python
requests.post("http://localhost:8501/v1/models/...")
```

### Flow

1. Convert input → JSON
2. Send HTTP POST
3. Receive predictions

**Key Terms**

* **REST API**: JSON-based prediction interface
* **Stateless Inference**: No session stored

---

## 6️⃣ gRPC Inference

```python
PredictRequest()
```

### Why gRPC?

* Faster than REST
* Used in **high-throughput systems**

**Key Terms**

* **gRPC**: Binary, low-latency RPC
* **Protocol Buffers**: Efficient data serialization

---

## 7️⃣ Deploying New Model Versions

```text
/my_mnist_model/
 ├── 0001
 ├── 0002
```

### Behavior

* TF Serving automatically switches to latest version
* Zero downtime deployment

**Key Terms**

* **Blue-Green Deployment**
* **Backward Compatibility**

---

## 8️⃣ Vertex AI Deployment

```python
aiplatform.Model.upload(...)
endpoint.deploy(...)
```

### What Vertex AI Provides?

* Fully managed serving
* Auto-scaling
* GPU/CPU support

**Key Terms**

* **Endpoint**: Public prediction service
* **Replica**: Serving instance
* **Auto-scaling**: Dynamic resource allocation

---

## 9️⃣ Batch Prediction

```python
mnist_model.batch_predict(...)
```

### Use Case

* Offline predictions on large datasets

**Key Terms**

* **Batch Inference**: Asynchronous predictions
* **GCS**: Google Cloud Storage

---

## 🔟 Mobile Deployment (TensorFlow Lite)

```python
converter = tf.lite.TFLiteConverter.from_saved_model(...)
```

### Why TFLite?

* Small size
* Optimized for mobile/edge devices

**Key Terms**

* **Model Quantization**: Reduce size & latency
* **Edge AI**: On-device inference

---

## 🌐 Web Deployment (TensorFlow.js)

* Run models directly in browser
* No backend required

**Key Terms**

* **Client-side inference**
* **Privacy-preserving AI**

---

## 🔥 Interview Summary Line

> “This chapter covers the full ML lifecycle—from training to scalable deployment using TensorFlow Serving, Vertex AI, batch prediction, and edge deployment with TFLite and TFJS.”

---

If you want next:

* ✅ **MCQs**
* ✅ **Scenario-based interview questions**
* ✅ **Architecture diagram explanation**
* ✅ **Production pitfalls & best practices**
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_04_1 page MCQ with answer</b>
</summary>
Below is a **1-page MCQ (Interview-ready)** based on **Chapter 19: Training & Deploying TensorFlow Models at Scale**
(Focused on code understanding, serving, deployment, and production AI concepts)

---

## 📘 AI Understanding – MCQs (Chapter 19)

### Q1. Why is `TF_USE_LEGACY_KERAS=1` set in this code?

A. To improve GPU performance
B. To force TensorFlow to use Keras 3
C. To ensure compatibility with older Keras APIs
D. To disable eager execution

---

### Q2. What is the purpose of saving the model in **SavedModel format**?

A. Faster training
B. Model compression
C. Deployment and serving compatibility
D. Visualization

---

### Q3. Which folder structure is REQUIRED by TensorFlow Serving?

A. `/model/version/variables`
B. `/model/version/saved_model.pb`
C. `/version/model/`
D. `/savedmodel/`

---

### Q4. Which port is used by TensorFlow Serving for REST API?

A. 5000
B. 8080
C. 8500
D. 8501

---

### Q5. What does `signature_name="serving_default"` indicate?

A. Default training signature
B. Default inference entry point
C. Default optimizer
D. Default loss function

---

### Q6. Why is `Rescaling(scale=1/255)` used in the model?

A. Normalize pixel values
B. Reduce overfitting
C. Convert labels
D. Speed up training

---

### Q7. What advantage does **gRPC** have over REST in TensorFlow Serving?

A. Easier debugging
B. Better browser support
C. Lower latency and binary protocol
D. No authentication needed

---

### Q8. What happens when a new model version (0002) is added?

A. Old version is deleted
B. Server restarts automatically
C. TensorFlow Serving loads it dynamically
D. Requests fail

---

### Q9. What is the role of **Vertex AI Endpoint**?

A. Model training
B. Batch preprocessing
C. Online prediction serving
D. Model conversion

---

### Q10. Which use-case best fits **Batch Prediction** on Vertex AI?

A. Real-time fraud detection
B. Streaming inference
C. Large offline predictions
D. Model monitoring

---

### Q11. Why convert a model to **TFLite**?

A. Improve accuracy
B. Enable cloud serving
C. Deploy on mobile/embedded devices
D. Support gRPC

---

### Q12. What does `Optimize.DEFAULT` do in TFLite conversion?

A. Prunes layers
B. Quantizes model automatically
C. Changes architecture
D. Freezes gradients

---

## ✅ Answer Key

1️⃣ C
2️⃣ C
3️⃣ B
4️⃣ D
5️⃣ B
6️⃣ A
7️⃣ C
8️⃣ C
9️⃣ C
🔟 C
1️⃣1️⃣ C
1️⃣2️⃣ B

---

## 🎯 Interview Tip (1-liner)

> “SavedModel + TensorFlow Serving enables scalable, versioned, production-ready inference across REST, gRPC, cloud, and edge.”

If you want next:
✔ Scenario-based questions
✔ Deployment flow diagram
✔ Vertex AI vs TF Serving comparison
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_05_Scenario-Based Interview Questions</b>
</summary>
Below is a **Scenario-Based Interview Q&A (1-page, concise)** mapped **directly to the Chapter-19 code** (Training & Deploying TF Models at Scale).

---

## Scenario-Based Interview Questions & Answers (AI / MLOps)

### 1️⃣ Scenario: Model works locally but fails in production

**Q:** Your TensorFlow model works in training but fails when deployed. What do you check first?
**A:**

* SavedModel signatures (`serving_default`)
* Input shape & dtype (e.g., `uint8` → `float`)
* Preprocessing layers inside the model
* TensorFlow / Keras version mismatch

---

### 2️⃣ Scenario: Keras 3 breaks legacy code

**Q:** Why is `TF_USE_LEGACY_KERAS=1` used?
**A:**
To force **Keras 2.x behavior** so older training & serving pipelines remain compatible.

---

### 3️⃣ Scenario: You need to serve a model at scale

**Q:** Why export model as `SavedModel` instead of `.h5`?
**A:**

* Required for **TensorFlow Serving**
* Supports versioning (`0001`, `0002`)
* Language-agnostic, production-ready format

---

### 4️⃣ Scenario: Client apps use different protocols

**Q:** When do you use REST vs gRPC?
**A:**

* **REST** → Simple clients, browsers, curl
* **gRPC** → Low latency, high throughput, microservices

---

### 5️⃣ Scenario: New model version must go live with zero downtime

**Q:** How does TF Serving handle this?
**A:**

* Versioned folders (`/my_model/0001`, `/0002`)
* Automatically loads latest version
* No service restart required

---

### 6️⃣ Scenario: High inference traffic spikes

**Q:** How does Vertex AI handle scaling?
**A:**

* Auto-scales replicas (`min_replica`, `max_replica`)
* GPU acceleration
* Load-balanced managed endpoints

---

### 7️⃣ Scenario: Need offline predictions for large datasets

**Q:** Why use Batch Prediction instead of Endpoint?
**A:**

* Cost-efficient
* No real-time latency needs
* Processes millions of records via GCS

---

### 8️⃣ Scenario: Model must run on mobile / IoT device

**Q:** Why convert to TensorFlow Lite?
**A:**

* Smaller binary
* Faster inference
* Hardware optimizations (quantization)

---

### 9️⃣ Scenario: Model accuracy drops after deployment

**Q:** What production checks do you add?
**A:**

* Input data drift monitoring
* Model version tracking
* Canary deployment (old + new model)

---

### 🔟 Scenario: Security concern in cloud deployment

**Q:** How do you secure Vertex AI access?
**A:**

* Service accounts
* IAM roles
* GCS bucket permissions
* Avoid embedding credentials in code

---

## 🔑 Key Concepts Interviewers Expect

* **SavedModel** – Production TF format
* **TF Serving** – High-performance inference server
* **Vertex AI Endpoint** – Managed scalable inference
* **Batch Prediction** – Offline large-scale inference
* **Model Versioning** – Zero-downtime upgrades
* **TFLite** – Edge/mobile deployment

---

### 💡 One-Line Interview Summary

> “This pipeline shows how to train once and deploy everywhere—cloud, edge, batch, and real-time—using TensorFlow’s production ecosystem.”

If you want next:
✅ **MCQs from this chapter**
✅ **Deployment architecture diagram explanation**
✅ **Failure & rollback scenarios**
            
</details>

# Using GPUs to Speed Up Computations

Let's check that TensorFlow can see the GPU:

In [ ]:
physical_gpus = tf.config.list_physical_devices("GPU")
physical_gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


If you want your TensorFlow script to use only GPUs \#0 and \#1 (based on PCI order), then you can set the environment variables `CUDA_DEVICE_ORDER=PCI_BUS_ID` and `CUDA_VISIBLE_DEVICES=0,1` before starting your script, or in the script itself before using TensorFlow.

## Managing the GPU RAM

To limit the amount of RAM to 2GB per GPU:

In [ ]:
#for gpu in physical_gpus:
#    tf.config.set_logical_device_configuration(
#        gpu,
#        [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
#    )

To make TensorFlow grab memory as it needs it (only releasing it when the process shuts down):

In [ ]:
#for gpu in physical_gpus:
#    tf.config.experimental.set_memory_growth(gpu, True)

Equivalently, you can set the `TF_FORCE_GPU_ALLOW_GROWTH` environment variable to `true` before using TensorFlow.

To split a physical GPU into two logical GPUs:

In [ ]:
#tf.config.set_logical_device_configuration(
#    physical_gpus[0],
#    [tf.config.LogicalDeviceConfiguration(memory_limit=2048),
#     tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
#)

In [ ]:
logical_gpus = tf.config.list_logical_devices("GPU")
logical_gpus

[LogicalDevice(name='/device:GPU:0', device_type='GPU')]


## Placing Operations and Variables on Devices

To log every variable and operation placement (this must be run just after importing TensorFlow):

In [ ]:
#tf.get_logger().setLevel("DEBUG")  # log level is INFO by default
#tf.debugging.set_log_device_placement(True)

In [ ]:
a = tf.Variable([1., 2., 3.])  # float32 variable goes to the GPU
a.device

'/job:localhost/replica:0/task:0/device:GPU:0'

In [ ]:
b = tf.Variable([1, 2, 3])  # int32 variable goes to the CPU
b.device

'/job:localhost/replica:0/task:0/device:CPU:0'

You can place variables and operations manually on the desired device using a `tf.device()` context:

In [ ]:
with tf.device("/cpu:0"):
    c = tf.Variable([1., 2., 3.])

c.device

'/job:localhost/replica:0/task:0/device:CPU:0'

If you specify a device that does not exist, or for which there is no kernel, TensorFlow will silently fallback to the default placement:

In [ ]:
# extra code

with tf.device("/gpu:1234"):
    d = tf.Variable([1., 2., 3.])

d.device

"'/job:localhost/replica:0/task:0/device:GPU:0'"

If you want TensorFlow to throw an exception when you try to use a device that does not exist, instead of falling back to the default device:

In [ ]:
tf.config.set_soft_device_placement(False)

# extra code
try:
    with tf.device("/gpu:1000"):
        d = tf.Variable([1., 2., 3.])
except tf.errors.InvalidArgumentError as ex:
    print(ex)

tf.config.set_soft_device_placement(True)  # extra code – back to soft placement

Could not satisfy device specification '/job:localhost/replica:0/task:0/device:GPU:1000'. enable_soft_placement=0. Supported device types [CPU]. All available devices [/job:localhost/replica:0/task:0/device:CPU:0].


## Parallel Execution Across Multiple Devices

If you want to set the number of inter-op or intra-op threads (this may be useful if you want to avoid saturating the CPU, or if you want to make TensorFlow single-threaded, to run a perfectly reproducible test case):

In [ ]:
#tf.config.threading.set_inter_op_parallelism_threads(10)
#tf.config.threading.set_intra_op_parallelism_threads(10)

# Training Models Across Multiple Devices

## Training at Scale Using the Distribution Strategies API

In [ ]:
# extra code – creates a CNN model for MNIST using Keras
def create_model():
    return tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])

In [ ]:
tf.random.set_seed(42)

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    model = create_model()  # create a Keras model normally
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])  # compile the model normally

batch_size = 100  # preferably divisible by the number of replicas
model.fit(X_train, y_train, epochs=10,
          validation_data=(X_valid, y_valid), batch_size=batch_size)

In [ ]:
type(model.weights[0])

tensorflow.python.distribute.values.MirroredVariable

In [ ]:
model.predict(X_new).round(2)  # extra code – the batch is split across all replicas

array([[0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)


In [ ]:
# extra code – shows that saving a model does not preserve its distribution
#              strategy
model.save("my_mirrored_model", save_format="tf")
model = tf.keras.models.load_model("my_mirrored_model")
type(model.weights[0])

INFO:tensorflow:Assets written to: my_mirrored_model/assets


tensorflow.python.ops.resource_variable_ops.ResourceVariable

In [ ]:
with strategy.scope():
    model = tf.keras.models.load_model("my_mirrored_model")

In [ ]:
type(model.weights[0])

tensorflow.python.distribute.values.MirroredVariable


If you want to specify the list of GPUs to use:

In [ ]:
strategy = tf.distribute.MirroredStrategy(devices=["/gpu:0", "/gpu:1"])

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


If you want to change the default all-reduce algorithm:

In [ ]:
strategy = tf.distribute.MirroredStrategy(
    cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


If you want to use the `CentralStorageStrategy`:

In [ ]:
strategy = tf.distribute.experimental.CentralStorageStrategy()

INFO:tensorflow:ParameterServerStrategy (CentralStorageStrategy if you are using a single machine) with compute_devices = ['/job:localhost/replica:0/task:0/device:CPU:0'], variable_device = '/job:localhost/replica:0/task:0/device:CPU:0'


In [ ]:
# To train on a TPU in Google Colab:
#if "google.colab" in sys.modules and "COLAB_TPU_ADDR" in os.environ:
#  tpu_address = "grpc://" + os.environ["COLAB_TPU_ADDR"]
#else:
#  tpu_address = ""
#resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu_address)
#tf.config.experimental_connect_to_cluster(resolver)
#tf.tpu.experimental.initialize_tpu_system(resolver)
#strategy = tf.distribute.experimental.TPUStrategy(resolver)

## Training a Model on a TensorFlow Cluster

A TensorFlow cluster is a group of TensorFlow processes running in parallel, usually on different machines, and talking to each other to complete some work, for example training or executing a neural network. Each TF process in the cluster is called a "task" (or a "TF server"). It has an IP address, a port, and a type (also called its role or its job). The type can be `"worker"`, `"chief"`, `"ps"` (parameter server) or `"evaluator"`:
* Each **worker** performs computations, usually on a machine with one or more GPUs.
* The **chief** performs computations as well, but it also handles extra work such as writing TensorBoard logs or saving checkpoints. There is a single chief in a cluster. If it is not defined, then it is worker #0.
* A **parameter server** (ps) only keeps track of variable values, it is usually on a CPU-only machine.
* The **evaluator** obviously takes care of evaluation. There is usually a single evaluator in a cluster.

The set of tasks that share the same type is often called a "job". For example, the "worker" job is the set of all workers.

To start a TensorFlow cluster, you must first define it. This means specifying all the tasks (IP address, TCP port, and type). For example, the following cluster specification defines a cluster with 3 tasks (2 workers and 1 parameter server). It's a dictionary with one key per job, and the values are lists of task addresses:

In [ ]:
cluster_spec = {
    "worker": [
        "machine-a.example.com:2222",     # /job:worker/task:0
        "machine-b.example.com:2222"      # /job:worker/task:1
    ],
    "ps": ["machine-a.example.com:2221"]  # /job:ps/task:0
}

Every task in the cluster may communicate with every other task in the server, so make sure to configure your firewall to authorize all communications between these machines on these ports (it's usually simpler if you use the same port on every machine).

When a task is started, it needs to be told which one it is: its type and index (the task index is also called the task id). A common way to specify everything at once (both the cluster spec and the current task's type and id) is to set the `TF_CONFIG` environment variable before starting the program. It must be a JSON-encoded dictionary containing a cluster specification (under the `"cluster"` key), and the type and index of the task to start (under the `"task"` key). For example, the following `TF_CONFIG` environment variable defines the same cluster as above, with 2 workers and 1 parameter server, and specifies that the task to start is worker \#0:

In [ ]:
os.environ["TF_CONFIG"] = json.dumps({
    "cluster": cluster_spec,
    "task": {"type": "worker", "index": 0}
})

Some platforms (e.g., Google Vertex AI) automatically set this environment variable for you.

TensorFlow's `TFConfigClusterResolver` class reads the cluster configuration from this environment variable:

In [ ]:
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
resolver.cluster_spec()

ClusterSpec({'ps': ['machine-a.example.com:2221'], 'worker': ['machine-a.example.com:2222', 'machine-b.example.com:2222']})

In [ ]:
resolver.task_type

'worker'

In [ ]:
resolver.task_id

0

Now let's run a simpler cluster with just two worker tasks, both running on the local machine. We will use the `MultiWorkerMirroredStrategy` to train a model across these two tasks.

The first step is to write the training code. As this code will be used to run both workers, each in its own process, we write this code to a separate Python file, `my_mnist_multiworker_task.py`. The code is relatively straightforward, but there are a couple important things to note:
* We create the `MultiWorkerMirroredStrategy` before doing anything else with TensorFlow.
* Only one of the workers will take care of logging to TensorBoard. As mentioned earlier, this worker is called the *chief*. When it is not defined explicitly, then by convention it is worker #0.

In [ ]:
%%writefile my_mnist_multiworker_task.py

import tempfile
import tensorflow as tf

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # at the start!
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()
print(f"Starting task {resolver.task_type} #{resolver.task_id}")

# extra code – Load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Rescaling(scale=1 / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)

if resolver.task_id == 0:  # the chief saves the model to the right location
    model.save("my_mnist_multiworker_model", save_format="tf")
else:
    tmpdir = tempfile.mkdtemp()  # other workers save to a temporary directory
    model.save(tmpdir, save_format="tf")
    tf.io.gfile.rmtree(tmpdir)  # and we can delete this directory at the end!

Writing my_mnist_multiworker_task.py


In a real world application, there would typically be a single worker per machine, but in this example we're running both workers on the same machine, so they will both try to use all the available GPU RAM (if this machine has a GPU), and this will likely lead to an Out-Of-Memory (OOM) error. To avoid this, we could use the `CUDA_VISIBLE_DEVICES` environment variable to assign a different GPU to each worker. Alternatively, we can simply disable GPU support, by setting `CUDA_VISIBLE_DEVICES` to an empty string.

We are now ready to start both workers, each in its own process. Notice that we change the task index:

In [ ]:
%%bash --bg

export CUDA_VISIBLE_DEVICES=''
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 0}}'
python my_mnist_multiworker_task.py > my_worker_0.log 2>&1

In [ ]:
%%bash --bg

export CUDA_VISIBLE_DEVICES=''
export TF_CONFIG='{"cluster": {"worker": ["127.0.0.1:9901", "127.0.0.1:9902"]},
                   "task": {"type": "worker", "index": 1}}'
python my_mnist_multiworker_task.py > my_worker_1.log 2>&1

**Note**: if you get warnings about `AutoShardPolicy`, you can safely ignore them. See [TF issue #42146](https://github.com/tensorflow/tensorflow/issues/42146) for more details.

That's it! Our TensorFlow cluster is now running, but we can't see it in this notebook because it's running in separate processes (but you can see the progress in `my_worker_*.log`).

Since the chief (worker #0) is writing to TensorBoard, we use TensorBoard to view the training progress. Run the following cell, then click on the settings button (i.e., the gear icon) in the TensorBoard interface and check the "Reload data" box to make TensorBoard automatically refresh every 30s. Once the first epoch of training is finished (which may take a few minutes), and once TensorBoard refreshes, the SCALARS tab will appear. Click on this tab to view the progress of the model's training and validation accuracy.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./my_mnist_multiworker_logs --port=6006

In [ ]:
# strategy = tf.distribute.MultiWorkerMirroredStrategy(
#     communication_options=tf.distribute.experimental.CommunicationOptions(
#         implementation=tf.distribute.experimental.CollectiveCommunication.NCCL))

## Running Large Training Jobs on Vertex AI

Let's copy the training script, but add `import os` and change the save path to be the GCS path that the `AIP_MODEL_DIR` environment variable will point to:

In [ ]:
%%writefile my_vertex_ai_training_task.py

import os
from pathlib import Path
import tempfile
import tensorflow as tf

strategy = tf.distribute.MultiWorkerMirroredStrategy()  # at the start!
resolver = tf.distribute.cluster_resolver.TFConfigClusterResolver()

if resolver.task_type == "chief":
    model_dir = os.getenv("AIP_MODEL_DIR")  # paths provided by Vertex AI
    tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")
    checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")
else:
    tmp_dir = Path(tempfile.mkdtemp())  # other workers use a temporary dirs
    model_dir = tmp_dir / "model"
    tensorboard_log_dir = tmp_dir / "logs"
    checkpoint_dir = tmp_dir / "ckpt"

callbacks = [tf.keras.callbacks.TensorBoard(tensorboard_log_dir),
             tf.keras.callbacks.ModelCheckpoint(checkpoint_dir)]

# extra code – Load and prepare the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – build and compile the Keras model using the distribution strategy
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28],
                                dtype=tf.uint8),
        tf.keras.layers.Lambda(lambda X: X / 255),
        tf.keras.layers.Conv2D(filters=64, kernel_size=7, activation="relu",
                               padding="same", input_shape=[28, 28, 1]),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu",
                               padding="same"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(units=10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2),
                  metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10,
          callbacks=callbacks)
model.save(model_dir, save_format="tf")

Writing my_vertex_ai_training_task.py


In [ ]:
custom_training_job = aiplatform.CustomTrainingJob(
    display_name="my_custom_training_job",
    script_path="my_vertex_ai_training_task.py",
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",
    model_serving_container_image_uri=server_image,
    requirements=["gcsfs==2022.3.0"],  # not needed, this is just an example
    staging_bucket=f"gs://{bucket_name}/staging"
)

In [ ]:
mnist_model2 = custom_training_job.run(
    machine_type="n1-standard-4",
    replica_count=2,
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,
)

Training script copied to:
gs://my_bucket/aiplatform-2022-04-14-10:08:24.124-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/aiplatform-custom-training-2022-04-14-10:08:25.226 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5407999068506947584?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_PENDING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/6685701948726837248?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/5407999068506947584 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-c

Let's clean up:

In [ ]:
mnist_model2.delete()
custom_training_job.delete()
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")
for blob in blobs:
    blob.delete()

# Hyperparameter Tuning on Vertex AI

In [ ]:
%%writefile my_vertex_ai_trial.py

import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--n_hidden", type=int, default=2)
parser.add_argument("--n_neurons", type=int, default=256)
parser.add_argument("--learning_rate", type=float, default=1e-2)
parser.add_argument("--optimizer", default="adam")
args = parser.parse_args()

import tensorflow as tf

def build_model(args):
    with tf.distribute.MirroredStrategy().scope():
        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))
        for _ in range(args.n_hidden):
            model.add(tf.keras.layers.Dense(args.n_neurons, activation="relu"))
        model.add(tf.keras.layers.Dense(10, activation="softmax"))
        opt = tf.keras.optimizers.get(args.optimizer)
        opt.learning_rate = args.learning_rate
        model.compile(loss="sparse_categorical_crossentropy", optimizer=opt,
                      metrics=["accuracy"])
        return model

# extra code – loads and splits the dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# extra code – use the AIP_* environment variable and create the callbacks
import os
model_dir = os.getenv("AIP_MODEL_DIR")
tensorboard_log_dir = os.getenv("AIP_TENSORBOARD_LOG_DIR")
checkpoint_dir = os.getenv("AIP_CHECKPOINT_DIR")
trial_id = os.getenv("CLOUD_ML_TRIAL_ID")
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)
callbacks = [tensorboard_cb, early_stopping_cb]

model = build_model(args)
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                    epochs=10, callbacks=callbacks)
model.save(model_dir, save_format="tf")  # extra code

import hypertune

hypertune = hypertune.HyperTune()
hypertune.report_hyperparameter_tuning_metric(
    hyperparameter_metric_tag="accuracy",  # name of the reported metric
    metric_value=max(history.history["val_accuracy"]),  # max accuracy value
    global_step=model.optimizer.iterations.numpy(),
)

Writing my_vertex_ai_trial.py


In [ ]:
trial_job = aiplatform.CustomJob.from_local_script(
    display_name="my_search_trial_job",
    script_path="my_vertex_ai_trial.py",  # path to your training script
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",
    staging_bucket=f"gs://{bucket_name}/staging",
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,  # in this example, each trial will have 2 GPUs
)

Training script copied to:
gs://homl3-mybucket5/staging/aiplatform-2022-04-18-18:14:02.860-aiplatform_custom_trainer_script-0.1.tar.gz.


In [ ]:
from google.cloud.aiplatform import hyperparameter_tuning as hpt

hp_job = aiplatform.HyperparameterTuningJob(
    display_name="my_hp_search_job",
    custom_job=trial_job,
    metric_spec={"accuracy": "maximize"},
    parameter_spec={
        "learning_rate": hpt.DoubleParameterSpec(min=1e-3, max=10, scale="log"),
        "n_neurons": hpt.IntegerParameterSpec(min=1, max=300, scale="linear"),
        "n_hidden": hpt.IntegerParameterSpec(min=1, max=10, scale="linear"),
        "optimizer": hpt.CategoricalParameterSpec(["sgd", "adam"]),
    },
    max_trial_count=100,
    parallel_trial_count=20,
)
hp_job.run()

Creating HyperparameterTuningJob
HyperparameterTuningJob created. Resource name: projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568
To use this HyperparameterTuningJob in another session:
hpt_job = aiplatform.HyperparameterTuningJob.get('projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568')
View HyperparameterTuningJob:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5825136187899117568?project=522977795627
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/522977795627/locations/us-central1/hyperparameterTuningJobs/5825136187899117568 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJ

In [ ]:
def get_final_metric(trial, metric_id):
    for metric in trial.final_measurement.metrics:
        if metric.metric_id == metric_id:
            return metric.value

trials = hp_job.trials
trial_accuracies = [get_final_metric(trial, "accuracy") for trial in trials]
best_trial = trials[np.argmax(trial_accuracies)]

In [ ]:
max(trial_accuracies)

0.977400004863739

In [ ]:
best_trial.id

'98'

In [ ]:
best_trial.parameters

[parameter_id: "learning_rate"
value {
  number_value: 0.001
}
, parameter_id: "n_hidden"
value {
  number_value: 8.0
}
, parameter_id: "n_neurons"
value {
  number_value: 216.0
}
, parameter_id: "optimizer"
value {
  string_value: "adam"
}
]

# Extra Material – Distributed Keras Tuner on Vertex AI

Instead of using Vertex AI's hyperparameter tuning service, you can use [Keras Tuner](https://keras.io/keras_tuner/) (introduced in Chapter 10) and run it on Vertex AI VMs. Keras Tuner provides a simple way to scale hyperparameter search by distributing it across multiple machines: it only requires setting three environment variables on each machine, then running your regular Keras Tuner code on each machine. You can use the exact same script on all machines. One of the machines acts as the chief, and the others act as workers. Each worker asks the chief which hyperparameter values to try—it acts as the oracle—then the worker trains the model using these hyperparameter values, and finally it reports the model's performance back to the chief, which can then decide which hyperparameter values the worker should try next.

The three environment variables you need to set on each machine are:

* `KERASTUNER_TUNER_ID`: equal to `"chief"` on the chief machine, or a unique identifier on each worker machine, such as `"worker0"`, `"worker1"`, etc.
* `KERASTUNER_ORACLE_IP`: the IP address or hostname of the chief machine. The chief itself should generally use `"0.0.0.0"` to listen on every IP address on the machine.
* `KERASTUNER_ORACLE_PORT`: the TCP port that the chief will be listening on.

You can use distributed Keras Tuner on any set of machines. If you want to run it on Vertex AI machines, then you can spawn a regular training job, and just modify the training script to set the three environment variables properly before using Keras Tuner.

For example, the script below starts by parsing the `TF_CONFIG` environment variable, which will be automatically set by Vertex AI, just like earlier. It finds the address of the task of type `"chief"`, and it extracts the IP address or hostname, and the TCP port. It then defines the tuner ID as the task type followed by the task index, for example `"worker0"`. If the tuner ID is `"chief0"`, it changes it to `"chief"`, and it sets the IP to `"0.0.0.0"`: this will make it listen on all IPv4 address on its machine. Then it defines the environment variables for Keras Tuner. Next, the script creates a tuner, just like in Chapter 10, the it runs the search, and finally it saves the best model to the location given by Vertex AI:

In [ ]:
%%writefile my_keras_tuner_search.py

import json
import os

tf_config = json.loads(os.environ["TF_CONFIG"])

chief_ip, chief_port = tf_config["cluster"]["chief"][0].rsplit(":", 1)
tuner_id = f'{tf_config["task"]["type"]}{tf_config["task"]["index"]}'
if tuner_id == "chief0":
    tuner_id = "chief"
    chief_ip = "0.0.0.0"
    # extra code – since the chief doesn't work much, you can optimize compute
    # resources by running a worker on the same machine. To do this, you can
    # just make the chief start another process, after tweaking the TF_CONFIG
    # environment variable to set the task type to "worker" and the task index
    # to a unique value. Uncomment the next few lines to give this a try:
    # import subprocess
    # import sys
    # tf_config["task"]["type"] = "workerX"  # the worker on the chief's machine
    # os.environ["TF_CONFIG"] = json.dumps(tf_config)
    # subprocess.Popen([sys.executable] + sys.argv,
    #                  stdout=sys.stdout, stderr=sys.stderr)

os.environ["KERASTUNER_TUNER_ID"] = tuner_id
os.environ["KERASTUNER_ORACLE_IP"] = chief_ip
os.environ["KERASTUNER_ORACLE_PORT"] = chief_port

from pathlib import Path
import keras_tuner as kt
import tensorflow as tf

gcs_path = "/gcs/my_bucket/my_hp_search"  # replace with your bucket's name

def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,
                             sampling="log")
    optimizer = hp.Choice("optimizer", values=["sgd", "adam"])
    if optimizer == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Flatten(input_shape=[28, 28], dtype=tf.uint8))
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])
    return model

hyperband_tuner = kt.Hyperband(
    build_model, objective="val_accuracy", seed=42,
    max_epochs=10, factor=3, hyperband_iterations=2,
    distribution_strategy=tf.distribute.MirroredStrategy(),
    directory=gcs_path, project_name="mnist")

# extra code – Load and split the MNIST dataset
mnist = tf.keras.datasets.mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = mnist
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

tensorboard_log_dir = os.environ["AIP_TENSORBOARD_LOG_DIR"] + "/" + tuner_id
tensorboard_cb = tf.keras.callbacks.TensorBoard(tensorboard_log_dir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)
hyperband_tuner.search(X_train, y_train, epochs=10,
                       validation_data=(X_valid, y_valid),
                       callbacks=[tensorboard_cb, early_stopping_cb])

if tuner_id == "chief":
    best_hp = hyperband_tuner.get_best_hyperparameters()[0]
    best_model = hyperband_tuner.hypermodel.build(best_hp)
    best_model.save(os.getenv("AIP_MODEL_DIR"), save_format="tf")

Writing my_keras_tuner_search.py


Note that Vertex AI automatically mounts the `/gcs` directory to GCS, using the open source [GCS Fuse adapter](https://cloud.google.com/storage/docs/gcs-fuse). This gives us a shared directory across the workers and the chief, which is required by Keras Tuner. Also note that we set the distribution strategy to a `MirroredStrategy`. This will allow each worker to use all the GPUs on its machine, if there's more than one.


Replace `/gcs/my_bucket/` with <code>/gcs/<i>{bucket_name}</i>/</code>:

In [ ]:
with open("my_keras_tuner_search.py") as f:
    script = f.read()

with open("my_keras_tuner_search.py", "w") as f:
    f.write(script.replace("/gcs/my_bucket/", f"/gcs/{bucket_name}/"))

Now all we need to do is to start a custom training job based on this script, exactly like in the previous section. Don't forget to add `keras-tuner` to the list of `requirements`:

In [ ]:
hp_search_job = aiplatform.CustomTrainingJob(
    display_name="my_hp_search_job",
    script_path="my_keras_tuner_search.py",
    container_uri="gcr.io/cloud-aiplatform/training/tf-gpu.2-4:latest",
    model_serving_container_image_uri=server_image,
    requirements=["keras-tuner~=1.1.2"],
    staging_bucket=f"gs://{bucket_name}/staging",
)

In [ ]:
mnist_model3 = hp_search_job.run(
    machine_type="n1-standard-4",
    replica_count=3,
    accelerator_type="NVIDIA_TESLA_K80",
    accelerator_count=2,
)

Training script copied to:
gs://my_bucket/staging/aiplatform-2022-04-15-13:34:32.591-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://my_bucket/staging/aiplatform-custom-training-2022-04-15-13:34:34.453 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/8601543785521872896?project=522977795627
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5022607048831926272?project=522977795627
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/522977795627/locations/us-central1/trainingPipelines/8601543785521872896 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomTrainingJob projects/52297779562

And we have a model!

Let's clean up:

In [ ]:
mnist_model3.delete()
hp_search_job.delete()
blobs = bucket.list_blobs(prefix=f"gs://{bucket_name}/staging/")
for blob in blobs:
    blob.delete()

# Extra Material – Using AutoML to Train a Model

Let's start by exporting the MNIST dataset to PNG images, and prepare an `import.csv` pointing to each image, and indicating the split (training, validation, or test) and the label:

In [ ]:
import matplotlib.pyplot as plt

mnist_path = Path("datasets/mnist")
mnist_path.mkdir(parents=True, exist_ok=True)
idx = 0
with open(mnist_path / "import.csv", "w") as import_csv:
    for split, X, y in zip(("training", "validation", "test"),
                           (X_train, X_valid, X_test),
                           (y_train, y_valid, y_test)):
        for image, label in zip(X, y):
            print(f"\r{idx + 1}/70000", end="")
            filename = f"{idx:05d}.png"
            plt.imsave(mnist_path / filename, np.tile(image, 3))
            line = f"{split},gs://{bucket_name}/mnist/{filename},{label}\n"
            import_csv.write(line)
            idx += 1

70000/70000

Let's upload this dataset to GCS:

In [ ]:
upload_directory(bucket, mnist_path)

Uploaded datasets/mnist                                              


Now let's create a managed image dataset on Vertex AI:

In [ ]:
from aiplatform.schema.dataset.ioformat.image import single_label_classification

mnist_dataset = aiplatform.ImageDataset.create(
    display_name="mnist-dataset",
    gcs_source=[f"gs://{bucket_name}/mnist/import.csv"],
    project=project_id,
    import_schema_uri=single_label_classification,
    sync=True,
)

Creating ImageDataset
Create ImageDataset backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3812233931370004480
ImageDataset created. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032
To use this ImageDataset in another session:
ds = aiplatform.ImageDataset('projects/522977795627/locations/us-central1/datasets/7532459492777132032')
Importing ImageDataset data: projects/522977795627/locations/us-central1/datasets/7532459492777132032
Import ImageDataset data backing LRO: projects/522977795627/locations/us-central1/datasets/7532459492777132032/operations/3010593197698056192
ImageDataset data imported. Resource name: projects/522977795627/locations/us-central1/datasets/7532459492777132032


Create an AutoML training job on this dataset:

**TODO**

<details>
<summary><b> ✅AI Understanding Template-ch0_01_Please provide answer for AI w.r.t. code explanation:                                   
    • What is it?
    • How does it reason?
    • Where does it fail?
    • When should I use it?
    • What is the mental model?
    • How do I prompt it?
    • What are alternatives? </b>
</summary>
Below is a **concise, interview-ready AI Understanding Template (Ch-0/1)** applied to the **given code block** (TensorFlow GPU, distributed training, Vertex AI).

---

## 1️⃣ What is it?

A **TensorFlow distributed training & deployment pipeline** that:

* Manages **GPU devices & memory**
* Trains models on **single/multiple GPUs**
* Scales training using **MirroredStrategy, MultiWorkerMirroredStrategy**
* Runs large jobs & hyperparameter tuning on **Google Vertex AI**

---

## 2️⃣ How does it reason?

* **Data parallelism**: same model replicated on GPUs/workers
* Each replica computes gradients → **All-Reduce** aggregates gradients
* **Synchronous SGD** updates shared weights
* Placement logic decides **CPU vs GPU kernels**
* Vertex AI orchestrates workers using **TF_CONFIG**

---

## 3️⃣ Where does it fail?

* GPU **OOM errors** if memory not managed
* Poor scaling if **batch size not divisible** by replicas
* Network bottlenecks in **multi-worker setups**
* Silent device fallback if **soft placement enabled**
* Debugging becomes hard across **multiple processes**

---

## 4️⃣ When should I use it?

Use when:

* Training is **slow on single GPU**
* Dataset/model is **large**
* Need **production-scale training**
* Running on **cloud (Vertex AI / TPU / multi-GPU VM)**

Avoid for:

* Small datasets
* Rapid prototyping
* CPU-only experimentation

---

## 5️⃣ What is the mental model?

> “One model, many workers.”

* Replicas = identical models
* Each replica processes **different data shard**
* Gradients are **synchronized**
* Chief handles **logging & checkpointing**
* Cloud infra handles **orchestration**

---

## 6️⃣ How do I prompt it? (as an AI user)

Good prompts:

* “Explain MirroredStrategy vs MultiWorkerMirroredStrategy”
* “Why does TensorFlow place float variables on GPU?”
* “How does TF_CONFIG enable distributed training?”
* “Debug OOM in multi-GPU TensorFlow”

Bad prompts:

* “Make it faster” (too vague)
* “Fix distributed training” (no context)

---

## 7️⃣ What are alternatives?

* **PyTorch Distributed (DDP)**
* **Horovod**
* **Ray Train**
* **JAX + pmap / pjrt**
* **Vertex AI AutoML**
* **Single-GPU + mixed precision**

---

## 🎯 One-line Interview Summary

> “This code shows how TensorFlow manages GPUs, distributes training across devices and machines, and scales model training and tuning using Vertex AI.”

If you want, I can also give:

* ✅ **1-page revision sheet**
* ✅ **MCQs from this code**
* ✅ **System design interview explanation**
* ✅ **Compare TF vs PyTorch distributed**

Just say 👍
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_02_Explain the code, get key terms and provide their full definition as well</b>
</summary>
Below is a **one-page AI Understanding Template (Ch-0/1)** style answer:
**Code explanation + key terms with full definitions**, written **interview-ready and concise**.

---

# AI Understanding Template – TensorFlow Device & Distributed Training

## 1️⃣ GPU Discovery & Configuration

### Code Explanation

```python
physical_gpus = tf.config.list_physical_devices("GPU")
```

* Lists **physical GPU hardware** available to TensorFlow.

```python
logical_gpus = tf.config.list_logical_devices("GPU")
```

* Lists **logical GPUs** created from physical GPUs (after memory limits/splitting).

### Key Terms

* **Physical GPU**: Actual hardware GPU detected by TensorFlow.
* **Logical GPU**: Virtual GPU created by splitting a physical GPU.
* **CUDA_VISIBLE_DEVICES**: Environment variable to control which GPUs TensorFlow can see.
* **PCI_BUS_ID**: Ensures GPU ordering matches hardware PCI order.

---

## 2️⃣ GPU Memory Management

### Code Explanation

```python
tf.config.set_logical_device_configuration(
    gpu,
    [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
)
```

* Limits GPU RAM usage to **2GB per GPU**.

```python
tf.config.experimental.set_memory_growth(gpu, True)
```

* Enables **on-demand memory allocation** instead of pre-allocating all GPU RAM.

### Key Terms

* **Memory Growth**: TensorFlow allocates GPU memory only when needed.
* **OOM (Out Of Memory)**: Error when GPU memory is exhausted.
* **TF_FORCE_GPU_ALLOW_GROWTH**: Env variable equivalent to memory growth.

---

## 3️⃣ Device Placement (CPU vs GPU)

### Code Explanation

```python
a = tf.Variable([1., 2., 3.])  # GPU
b = tf.Variable([1, 2, 3])    # CPU
```

* Float tensors → GPU
* Integer tensors → CPU (default behavior)

```python
with tf.device("/cpu:0"):
    c = tf.Variable([1., 2., 3.])
```

* Manually forces placement on CPU.

```python
tf.config.set_soft_device_placement(False)
```

* Throws error if TensorFlow cannot place an op on requested device.

### Key Terms

* **Device Placement**: Where TensorFlow executes an operation (CPU/GPU/TPU).
* **Soft Placement**: Falls back silently to another device.
* **Hard Placement**: Raises error if device is unavailable.

---

## 4️⃣ Parallelism & Threading

### Code Explanation

```python
tf.config.threading.set_inter_op_parallelism_threads(10)
tf.config.threading.set_intra_op_parallelism_threads(10)
```

* Controls CPU thread usage for reproducibility and performance tuning.

### Key Terms

* **Inter-op Parallelism**: Parallel execution of independent operations.
* **Intra-op Parallelism**: Parallel execution within a single operation.

---

## 5️⃣ Multi-GPU Training (MirroredStrategy)

### Code Explanation

```python
strategy = tf.distribute.MirroredStrategy()
```

* Replicates model on **each GPU** and synchronizes gradients.

```python
with strategy.scope():
    model = create_model()
```

* Ensures variables are **distributed variables**.

### Key Terms

* **MirroredStrategy**: Synchronous multi-GPU training on one machine.
* **Replica**: One copy of the model per device.
* **All-Reduce**: Algorithm to aggregate gradients across GPUs.

---

## 6️⃣ Distributed Training (Multi-Worker)

### Code Explanation

```python
tf.distribute.MultiWorkerMirroredStrategy()
```

* Extends MirroredStrategy across **multiple machines**.

```python
TF_CONFIG
```

* Environment variable defining cluster topology.

### Key Terms

* **Chief**: Handles checkpoints, logs, model saving.
* **Worker**: Performs training computation.
* **Parameter Server (PS)**: Stores and updates model parameters.
* **TF_CONFIG**: JSON config defining cluster roles & addresses.

---

## 7️⃣ TPU & Cloud Training (Vertex AI)

### Code Explanation

* Uses **TPUStrategy** or **MultiWorkerMirroredStrategy**
* Vertex AI auto-sets `TF_CONFIG`
* Models saved to **GCS paths**

### Key Terms

* **TPU**: Tensor Processing Unit, optimized for deep learning.
* **Vertex AI**: Google Cloud managed ML platform.
* **GCS Fuse**: Mounts cloud storage as local filesystem.

---

## 8️⃣ Hyperparameter Tuning

### Code Explanation

* Uses **Vertex AI HPT** or **Keras Tuner**
* Multiple trials run in parallel
* Best model selected based on validation metric

### Key Terms

* **Hyperparameter**: Config not learned by model (LR, layers, neurons).
* **Trial**: One training run with specific hyperparameters.
* **Search Space**: Range of values to explore.
* **Early Stopping**: Stops training when validation stops improving.

---

## 🎯 Interview One-Liner

> “This code demonstrates TensorFlow’s end-to-end hardware management, from GPU memory control and device placement to scalable distributed training across GPUs, machines, and cloud infrastructure using strategy APIs.”

If you want:
✔ **Diagram version**
✔ **MCQs from this code**
✔ **1-minute oral explanation**
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_03_cheatsheet</b>
</summary>
Below is a **1-page AI Interview Cheat Sheet (Code Explanation)** for **TensorFlow GPU, Distributed Training, and Vertex AI**.
(Concise, concept-focused, interview-ready)

---

## AI Understanding Template – GPU & Distributed Training (Cheat Sheet)

---

### 1️⃣ GPU Detection & Control

```python
tf.config.list_physical_devices("GPU")
```

✔ Detects available **physical GPUs**
✔ Used to verify CUDA/GPU setup

**Environment variables**

* `CUDA_VISIBLE_DEVICES=0,1` → restricts visible GPUs
* `CUDA_DEVICE_ORDER=PCI_BUS_ID` → stable GPU ordering

---

### 2️⃣ GPU Memory Management

**Problem:** TensorFlow may allocate all GPU memory upfront.

✔ Limit GPU memory:

```python
tf.config.LogicalDeviceConfiguration(memory_limit=2048)
```

✔ Enable dynamic growth:

```python
tf.config.experimental.set_memory_growth(gpu, True)
```

✔ Split one GPU into logical GPUs → useful for experimentation

---

### 3️⃣ Device Placement (CPU vs GPU)

```python
a = tf.Variable([1.,2.,3.])  # float → GPU
b = tf.Variable([1,2,3])    # int → CPU
```

✔ TensorFlow auto-places ops based on kernel availability
✔ Manual placement:

```python
with tf.device("/cpu:0"):
```

✔ Invalid device → **silent fallback**
✔ Strict mode:

```python
tf.config.set_soft_device_placement(False)
```

---

### 4️⃣ Parallelism (CPU Threads)

```python
tf.config.threading.set_inter_op_parallelism_threads()
tf.config.threading.set_intra_op_parallelism_threads()
```

✔ Controls CPU parallel execution
✔ Useful for reproducibility & performance tuning

---

### 5️⃣ Multi-GPU Training (Single Machine)

```python
strategy = tf.distribute.MirroredStrategy()
```

✔ Synchronous data-parallel training
✔ One model copy per GPU
✔ Gradients are **all-reduced**

⚠ Must create & compile model **inside `strategy.scope()`**

---

### 6️⃣ Saving & Loading Distributed Models

✔ Saved model **does NOT remember** distribution strategy
✔ Must re-wrap loading in `strategy.scope()`

---

### 7️⃣ Multi-Worker Training (Multiple Machines)

```python
tf.distribute.MultiWorkerMirroredStrategy()
```

✔ Uses `TF_CONFIG` to define cluster
✔ Roles:

* **chief** → saves model, logs
* **worker** → training
* **ps** → parameter server (optional)

---

### 8️⃣ Vertex AI Training

✔ Uses same TF distributed concepts
✔ Environment variables provided automatically:

* `AIP_MODEL_DIR`
* `AIP_TENSORBOARD_LOG_DIR`

✔ Custom training job:

```python
aiplatform.CustomTrainingJob(...)
```

✔ Scales to multiple machines + GPUs

---

### 9️⃣ Hyperparameter Tuning (Vertex AI)

✔ Parallel trials with different parameters
✔ Reports metrics using:

```python
hypertune.report_hyperparameter_tuning_metric()
```

✔ Supports:

* Learning rate
* Neurons
* Layers
* Optimizer

---

### 🔟 Keras Tuner (Distributed)

✔ Chief = oracle
✔ Workers request trials
✔ Requires:

* `KERASTUNER_TUNER_ID`
* `KERASTUNER_ORACLE_IP`
* `KERASTUNER_ORACLE_PORT`

✔ Uses shared GCS (`/gcs/`) for synchronization

---

### 🎯 Interview One-Liner

> “This code shows how TensorFlow manages GPU memory, device placement, and scales training from single-GPU to multi-GPU, multi-worker clusters, including Vertex AI production workflows and hyperparameter tuning.”

---

If you want:
✔ **MCQs from this**
✔ **Diagram explanation**
✔ **Beginner vs Production comparison**
✔ **Only GPU-related summary**

Just say 👍
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_04_1 page MCQ with answer</b>
</summary>
Below is a **1-page MCQ (AI Understanding – Ch 0/1)** focused on **TensorFlow GPU management, distributed training, and Vertex AI**, strictly based on your code.

---

## 📘 MCQs: TensorFlow GPU, Distribution & Vertex AI

### Q1. What does `tf.config.list_physical_devices("GPU")` return?

A. Logical GPU devices
B. Available CUDA cores
C. Physical GPU devices visible to TensorFlow
D. GPU memory usage

---

### Q2. What is the purpose of `CUDA_VISIBLE_DEVICES=0,1`?

A. Limits GPU memory
B. Enables multi-threading
C. Selects which GPUs TensorFlow can see
D. Changes GPU execution order

---

### Q3. What does `set_memory_growth(gpu, True)` do?

A. Allocates all GPU memory at startup
B. Frees GPU memory after each step
C. Allocates GPU memory on demand
D. Disables GPU usage

---

### Q4. What happens if TensorFlow is asked to place an op on a non-existent GPU?

A. Program crashes always
B. TensorFlow silently falls back to CPU
C. GPU is created virtually
D. Model stops training

---

### Q5. Which setting forces TensorFlow to raise an error instead of falling back?

A. `tf.debugging.set_log_device_placement(True)`
B. `tf.config.set_soft_device_placement(False)`
C. `CUDA_DEVICE_ORDER`
D. `TF_FORCE_GPU_ALLOW_GROWTH`

---

### Q6. Why does `float32` variable go to GPU but `int32` goes to CPU?

A. GPU doesn’t support integers
B. TensorFlow prefers GPU for floating-point ops
C. CPU is faster for int ops
D. It is random behavior

---

### Q7. What is `tf.distribute.MirroredStrategy()` used for?

A. Model parallelism across layers
B. Asynchronous training
C. Synchronous data-parallel training on multiple GPUs
D. TPU-only training

---

### Q8. Why should `batch_size` be divisible by number of replicas?

A. To reduce overfitting
B. For equal data split across replicas
C. To avoid NaN loss
D. To increase learning rate

---

### Q9. After saving a model trained with `MirroredStrategy`, what happens on reload?

A. Strategy is preserved
B. Strategy is lost
C. Model becomes non-trainable
D. Weights are reset

---

### Q10. What is the role of the **chief** in multi-worker training?

A. Performs no computation
B. Only evaluates model
C. Saves checkpoints and logs
D. Hosts parameter server

---

### Q11. What environment variable defines cluster configuration?

A. `CUDA_VISIBLE_DEVICES`
B. `KERASTUNER_TUNER_ID`
C. `TF_CONFIG`
D. `AIP_MODEL_DIR`

---

### Q12. Why disable GPU (`CUDA_VISIBLE_DEVICES=''`) in local multi-worker demo?

A. GPUs are slower
B. To avoid GPU memory conflicts
C. TPU compatibility
D. Reduce model size

---

### Q13. What does Vertex AI automatically provide to training jobs?

A. GPUs only
B. TF_CONFIG and GCS paths
C. Model architecture
D. Hyperparameters

---

### Q14. Purpose of `AIP_MODEL_DIR`?

A. Temporary cache
B. Dataset location
C. Where trained model must be saved
D. Logging directory

---

### Q15. What does Hyperparameter Tuning optimize in Vertex AI?

A. Training speed
B. GPU memory
C. User-defined metrics (e.g., accuracy)
D. Dataset size

---

## ✅ Answers

1️⃣ C
2️⃣ C
3️⃣ C
4️⃣ B
5️⃣ B
6️⃣ B
7️⃣ C
8️⃣ B
9️⃣ B
🔟 C
1️⃣1️⃣ C
1️⃣2️⃣ B
1️⃣3️⃣ B
1️⃣4️⃣ C
1️⃣5️⃣ C

---

### 🎯 Interview Tip (1-liner)

> “This code demonstrates **GPU control, fault-tolerant device placement, and scalable distributed training using TensorFlow strategies and Vertex AI**.”

If you want:
✔ **Scenario-based Q&A**
✔ **Whiteboard explanation version**
✔ **Beginner vs Advanced MCQ split**
            
</details>

<details>
<summary><b> ✅AI Understanding Template-ch0_05_Scenario-Based Interview Questions</b>
</summary>
Below is a **concise, scenario-based AI interview explanation** of the code.
(Short, structured, and concept-focused as requested.)

---

## 1️⃣ GPU Discovery & Control

```python
tf.config.list_physical_devices("GPU")
```

**Why:** Detect available GPUs before TensorFlow initializes them.
**Interview point:** Always query devices *before* allocating memory or building models.

---

## 2️⃣ GPU Selection (CUDA_VISIBLE_DEVICES)

```bash
CUDA_VISIBLE_DEVICES=0,1
```

**Why:** Restricts TensorFlow to specific GPUs.
**Use case:** Multi-user servers or isolating experiments.

---

## 3️⃣ GPU Memory Management

### Fixed memory

```python
LogicalDeviceConfiguration(memory_limit=2048)
```

**Why:** Prevents OOM by capping GPU RAM.

### Memory growth

```python
set_memory_growth(gpu, True)
```

**Why:** TensorFlow allocates GPU memory *on demand*.
**Interview answer:** Preferred when sharing GPUs.

---

## 4️⃣ Logical GPUs

```python
set_logical_device_configuration()
```

**Why:** Split one physical GPU into multiple logical GPUs.
**Use case:** Simulate multi-GPU training on one card.

---

## 5️⃣ Device Placement (CPU vs GPU)

```python
a = tf.Variable([1.,2.,3.])  # GPU
b = tf.Variable([1,2,3])    # CPU
```

**Reason:**

* `float32` → GPU kernels available
* `int32` → CPU fallback

---

## 6️⃣ Manual Device Placement

```python
with tf.device("/cpu:0"):
```

**Why:** Force operations to specific hardware (debugging, memory control).

---

## 7️⃣ Soft vs Hard Device Placement

```python
tf.config.set_soft_device_placement(False)
```

**Soft (default):** Silent fallback
**Hard:** Throws error if device unavailable
**Interview tip:** Use hard placement for debugging infra issues.

---

## 8️⃣ Multi-GPU Training (MirroredStrategy)

```python
strategy = tf.distribute.MirroredStrategy()
```

**What happens:**

* Model copied to each GPU
* Gradients synced via **All-Reduce**
* Single logical model

**Best practice:** `batch_size` divisible by number of GPUs.

---

## 9️⃣ Model Saving & Strategy

```python
model.save()
```

**Key point:**
Saved models **do NOT** store distribution strategy.
Must reload inside `strategy.scope()`.

---

## 🔟 Alternative Distribution Strategies

| Strategy            | Use Case                    |
| ------------------- | --------------------------- |
| MirroredStrategy    | Single machine, multi-GPU   |
| MultiWorkerMirrored | Multi-machine sync training |
| CentralStorage      | Variables on CPU            |
| TPUStrategy         | TPU training                |

---

## 1️⃣1️⃣ TF_CONFIG & Cluster Training

```json
TF_CONFIG = {
  "cluster": {...},
  "task": {"type":"worker","index":0}
}
```

**Why:** Defines distributed cluster topology.
**Roles:** worker, chief, ps, evaluator.

---

## 1️⃣2️⃣ Multi-Worker Training

```python
MultiWorkerMirroredStrategy()
```

**Scenario:** Same training script runs on all machines.
Chief handles:

* Checkpoints
* TensorBoard
* Model saving

---

## 1️⃣3️⃣ Vertex AI Custom Training

**Why Vertex AI:**

* Managed infra
* Auto-scaling
* Native TensorBoard
* GCS integration

Environment variables:

* `AIP_MODEL_DIR`
* `AIP_TENSORBOARD_LOG_DIR`

---

## 1️⃣4️⃣ Hyperparameter Tuning (Vertex AI)

```python
HyperparameterTuningJob
```

**Concept:**

* Parallel trials
* Metric-driven search
* Supports grid, random, Bayesian

---

## 1️⃣5️⃣ Distributed Keras Tuner

**Key idea:**
Chief = Oracle
Workers = Trainers
Shared storage via **GCS Fuse**

---

## 🎯 Interview One-Liner

> “This code demonstrates end-to-end scalable TensorFlow training—from GPU memory control, device placement, and mirrored multi-GPU training to distributed multi-worker clusters and Vertex AI hyperparameter tuning.”

---

If you want next:

* ✅ **1-page revision sheet**
* ✅ **MCQs from this code**
* ✅ **Real interview questions + answers**
            
</details>

# Exercise Solutions

## 1. to 8.

1. A SavedModel contains a TensorFlow model, including its architecture (a computation graph) and its weights. It is stored as a directory containing a _saved_model.pb_ file, which defines the computation graph (represented as a serialized protocol buffer), and a _variables_ subdirectory containing the variable values. For models containing a large number of weights, these variable values may be split across multiple files. A SavedModel also includes an _assets_ subdirectory that may contain additional data, such as vocabulary files, class names, or some example instances for this model. To be more accurate, a SavedModel can contain one or more _metagraphs_. A metagraph is a computation graph plus some function signature definitions (including their input and output names, types, and shapes). Each metagraph is identified by a set of tags. To inspect a SavedModel, you can use the command-line tool `saved_model_cli` or just load it using `tf.saved_model.load()` and inspect it in Python.
2. TF Serving allows you to deploy multiple TensorFlow models (or multiple versions of the same model) and make them accessible to all your applications easily via a REST API or a gRPC API. Using your models directly in your applications would make it harder to deploy a new version of a model across all applications. Implementing your own microservice to wrap a TF model would require extra work, and it would be hard to match TF Serving's features. TF Serving has many features: it can monitor a directory and autodeploy the models that are placed there, and you won't have to change or even restart any of your applications to benefit from the new model versions; it's fast, well tested, and scales very well; and it supports A/B testing of experimental models and deploying a new model version to just a subset of your users (in this case the model is called a _canary_). TF Serving is also capable of grouping individual requests into batches to run them jointly on the GPU. To deploy TF Serving, you can install it from source, but it is much simpler to install it using a Docker image. To deploy a cluster of TF Serving Docker images, you can use an orchestration tool such as Kubernetes, or use a fully hosted solution such as Google Vertex AI.
3. To deploy a model across multiple TF Serving instances, all you need to do is configure these TF Serving instances to monitor the same _models_ directory, and then export your new model as a SavedModel into a subdirectory.
4. The gRPC API is more efficient than the REST API. However, its client libraries are not as widely available, and if you activate compression when using the REST API, you can get almost the same performance. So, the gRPC API is most useful when you need the highest possible performance and the clients are not limited to the REST API.
5. To reduce a model's size so it can run on a mobile or embedded device, TFLite uses several techniques:
    * It provides a converter which can optimize a SavedModel: it shrinks the model and reduces its latency. To do this, it prunes all the operations that are not needed to make predictions (such as training operations), and it optimizes and fuses operations whenever possible.
    * The converter can also perform post-training quantization: this technique dramatically reduces the model’s size, so it’s much faster to download and store.
    * It saves the optimized model using the FlatBuffer format, which can be loaded to RAM directly, without parsing. This reduces the loading time and memory footprint.
6. Quantization-aware training consists in adding fake quantization operations to the model during training. This allows the model to learn to ignore the quantization noise; the final weights will be more robust to quantization.
7. Model parallelism means chopping your model into multiple parts and running them in parallel across multiple devices, hopefully speeding up the model during training or inference. Data parallelism means creating multiple exact replicas of your model and deploying them across multiple devices. At each iteration during training, each replica is given a different batch of data, and it computes the gradients of the loss with regard to the model parameters. In synchronous data parallelism, the gradients from all replicas are then aggregated and the optimizer performs a Gradient Descent step. The parameters may be centralized (e.g., on parameter servers) or replicated across all replicas and kept in sync using AllReduce. In asynchronous data parallelism, the parameters are centralized and the replicas run independently from each other, each updating the central parameters directly at the end of each training iteration, without having to wait for the other replicas. To speed up training, data parallelism turns out to work better than model parallelism, in general. This is mostly because it requires less communication across devices. Moreover, it is much easier to implement, and it works the same way for any model, whereas model parallelism requires analyzing the model to determine the best way to chop it into pieces. That said, research in this domain is making quick progress (e.g., PipeDream or Pathways), so a mix of model parallelism and data parallelism is probably the way forward.
8. When training a model across multiple servers, you can use the following distribution strategies:
    * The `MultiWorkerMirroredStrategy` performs mirrored data parallelism. The model is replicated across all available servers and devices, and each replica gets a different batch of data at each training iteration and computes its own gradients. The mean of the gradients is computed and shared across all replicas using a distributed AllReduce implementation (NCCL by default), and all replicas perform the same Gradient Descent step. This strategy is the simplest to use since all servers and devices are treated in exactly the same way, and it performs fairly well. In general, you should use this strategy. Its main limitation is that it requires the model to fit in RAM on every replica.
    * The `ParameterServerStrategy` performs asynchronous data parallelism. The model is replicated across all devices on all workers, and the parameters are sharded across all parameter servers. Each worker has its own training loop, running asynchronously with the other workers; at each training iteration, each worker gets its own batch of data and fetches the latest version of the model parameters from the parameter servers, then it computes the gradients of the loss with regard to these parameters, and it sends them to the parameter servers. Lastly, the parameter servers perform a Gradient Descent step using these gradients. This strategy is generally slower than the previous strategy, and a bit harder to deploy, since it requires managing parameter servers. However, it can be useful in some situations, especially when you can take advantage of the asynchronous updates, for example to reduce I/O bottlenecks. This depends on many factors, including hardware, network topology, number of servers, model size, and more, so your mileage may vary.

## 9.
_Exercise: Train a model (any model you like) and deploy it to TF Serving or Google Vertex AI. Write the client code to query it using the REST API or the gRPC API. Update the model and deploy the new version. Your client code will now query the new version. Roll back to the first version._

Please follow the steps in the <a href="#Deploying-TensorFlow-models-to-TensorFlow-Serving-(TFS)">Deploying TensorFlow models to TensorFlow Serving</a> section above.

# 10.
_Exercise: Train any model across multiple GPUs on the same machine using the `MirroredStrategy` (if you do not have access to GPUs, you can use Colaboratory with a GPU Runtime and create two virtual GPUs). Train the model again using the `CentralStorageStrategy `and compare the training time._

Please follow the steps in the [Distributed Training](#Distributed-Training) section above.

# 11.
_Exercise: Train a small model on Google Vertex AI, using TensorFlow Cloud Tuner for hyperparameter tuning._

Please follow the instructions in the _Hyperparameter Tuning using TensorFlow Cloud Tuner_ section in the book.

# Congratulations!

You've reached the end of the book! I hope you found it useful. 😊